# Modellierung nichtlinearer Einzelhandelsnachfragekurven mit PROC GAM

## Zusammenfassung

Dieses Notebook nutzt **PROC GAM**, um wöchentliche Lebensmittel-Stückverkäufe als glatte, nichtlineare Funktion von Regalpreis, Filialtemperatur (ein Saisonalitäts-Proxy) und Werbeausgaben zu modellieren, mit einem parametrischen Aktions-Effekt. Generalisierte additive Modelle erlauben es einem Kategoriemanager, die echten, gekrümmten Preiselastizitäts- und saisonalen Nachfrageformen aufzudecken, die eine lineare Regression übersehen würde, und unterstützen so schärfere Preis- und Aktionsentscheidungen.

## Datenquellen

| Datensatz | Zeilen | Granularität | Schlüsselvariablen | Beschreibung |
|---------|------|-------|---------------|-------------|
| `store_sales` | 100 | Woche | `Week`, `Price`, `Temperature`, `Promotion`, `PromoSpend`, `Units` | Synthetische wöchentliche Kassendaten für eine Flaggschiff-Filiale über 100 aufeinanderfolgende Wochen (fast zwei Saisonzyklen). Inline erzeugt mit `call streaminit(20250531)` und `rand()`. Die wöchentlichen Stückverkäufe folgen einem Poisson-Nachfrageprozess, dessen Mittelwert durch eine exponentielle Preisreaktionskurve, einen quadratischen Temperatur-/Saisonalitätseffekt mit Höhepunkt nahe 72 °F und eine konkave (Quadratwurzel-)Werbeausgaben-Wirkung getrieben wird, zuzüglich eines diskreten Aktionswochen-Kennzeichens. |

# Modellierung nichtlinearer Einzelhandelsnachfragekurven mit PROC GAM

Die Nachfrage im Einzelhandel reagiert selten linear auf Preis, Wetter oder Werbeausgaben. Eine kleine Preissenkung bei einem Grundnahrungsmittel bewegt das Volumen kaum, während das Überschreiten einer psychologischen Preisschwelle einen sprunghaften Anstieg auslösen kann; wetterbedingte Nachfrage erreicht ihren Höhepunkt in einem angenehmen Mittelbereich und fällt an beiden Extremen ab; und die Wirkung von Werbeausgaben zeigt abnehmende Grenzerträge, je höher die Ausgaben steigen.

**PROC GAM** (generalisierte additive Modelle) passt jeden Treiber mit einem eigenen glatten Spline-Term an, sodass die Daten — nicht eine starre lineare Annahme — die Form jeder Nachfragekurve bestimmen. Hier modellieren wir die wöchentlichen Stückverkäufe einer einzelnen Flaggschiff-Lebensmittelfiliale über 100 aufeinanderfolgende Wochen und kombinieren ein parametrisches Aktions-Kennzeichen mit Glättungssplines für Preis, Temperatur und Werbeausgaben unter einer Poisson-Antwort.

## Schritt 1 — Erzeugen einer synthetischen wöchentlichen Verkaufsserie

Wir simulieren 100 aufeinanderfolgende Wochen (fast zwei Saisonzyklen) an Kassendaten für eine Flaggschiff-Filiale. Der datengenerierende Prozess ist bewusst nichtlinear, damit wir bestätigen können, dass GAM realistische Formen wiederherstellt:

- **Price** treibt das Volumen über eine exponentielle Reaktionskurve (`exp(-1.1 * Price)`), sodass die Nachfrage steil steigt, wenn der Preis fällt.
- **Temperature** wirkt als Saisonalitäts-Proxy mit einem quadratischen Höhepunkt nahe 72 °F und bildet komfortgetriebenen Kundenverkehr ab.
- **PromoSpend** liefert eine konkave Quadratwurzel-Wirkung (abnehmende Grenzerträge).
- Ein diskretes **Promotion**-Kennzeichen fügt einen parametrischen Stufeneffekt in beworbenen Wochen hinzu.

Die wöchentlichen `Units` werden aus einer Poisson-Verteilung gezogen, passend zur Zählnatur der Stückverkäufe.

In [1]:
DATEN store_sales;
   AUFRUFEN streaminit(20250531);
   LÄNGE Promotion $3;
   AUSFÜHRUNG Week = 1 BIS 100;
      BasePrice = 3.20 + 0.30 * rand("uniform");
      Discount  = 0.40 * rand("uniform");
      Price     = round(BasePrice - Discount, 0.01);
      WENN rand("uniform") < 0.28 DANN AUSFÜHRUNG;
         Promotion  = "Yes";
         PromoSpend = round(200 + 600 * rand("uniform"), 1);
      ENDE;
      SONST AUSFÜHRUNG;
         Promotion  = "No";
         PromoSpend = 0;
      ENDE;
      Temperature = 55 + 25 * sin((Week - 12) / 52 * 2 * 3.14159265)
                    + 4 * rand("normal");
      priceEffect = 180 * EXP(-1.1 * Price);
      tempEffect  = -0.012 * (Temperature - 72) ** 2;
      promoEffect = 0.085 * sqrt(PromoSpend);
      logMu = 3.0 + LOG(priceEffect) + tempEffect + promoEffect;
      Units = rand("poisson", EXP(logMu) / 12);
      AUSGABE;
   ENDE;
AUSFÜHREN;



NOTE: DATA store_sales


NOTE: Wrote store_sales (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.02 seconds
  cpu   0.02 seconds


## Schritt 2 — Profilierung der simulierten Daten

Ein schnelles `PROC MEANS` bestätigt, dass die Variablen sinnvolle Einzelhandelsbereiche abdecken, bevor modelliert wird: Stückzahlen sind nicht-negative ganze Zahlen, der Preis gruppiert sich um wenige Dollar, die Temperatur durchläuft einen vollen Saisonzyklus, und die Werbeausgaben sind in nicht beworbenen Wochen null.

In [2]:
PROZEDUR MITTELWERTE DATEN=store_sales n mean MIN MAX maxdec=2;
   VAR Units Price Temperature PromoSpend;
   BEZEICHNUNG Units="Verkaufte Einheiten" Price="Preis ($)"
         Temperature="Temperatur (°F)" PromoSpend="Werbeausgaben ($)";
AUSFÜHREN;


                                                  The MEANS Procedure

 Variable     Label                       N           Mean     Minimum     Maximum
 ---------------------------------------------------------------------------------
 Units        Verkaufte Einheiten       100           7.03        0.00      103.00
 Price        Preis ($)                 100           3.15        2.81        3.48
 Temperature  Temperatur (°F)           100          55.50       22.72       83.49
 PromoSpend   Werbeausgaben ($)         100         128.76        0.00      779.00
 ---------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Schritt 3 — Anpassung des vollständigen additiven Nachfragemodells

Das Kernmodell kombiniert:

- `param(Promotion)` — ein parametrischer (linearer) Effekt für den Aktionswochen-Indikator, deklariert in der `CLASS`-Anweisung.
- `spline(Price, df=4)` — ein kubischer Glättungsspline, der die gekrümmte Preisreaktion erfasst.
- `spline(Temperature, df=4)` — eine glatte Saisonalitätskurve.
- `spline(PromoSpend, df=3)` — Werbewirkung mit abnehmenden Grenzerträgen.

Da die Stückverkäufe Zählungen sind, geben wir `dist=poisson` (Log-Link) an. `method=gcv` lässt die generalisierte Kreuzvalidierung die Glättung jeder Komponente steuern, ohne eine explizite Freiheitsgrad-Vorgabe. Die `OUTPUT`-Anweisung schreibt Vorhersagen und Residuen pro Beobachtung nach `gam_fit`.

Die Prozedur berichtet eine **Devianz von 43,59** gegenüber einer **Null-Devianz von 2041,12** — die vier glatten plus parametrischen Treiber erklären fast die gesamte Variation, die ein reines Konstantenmodell übrig lässt — und ein **AIC von 254,61**. Die parametrische `PROMOTIONYES`-Schätzung ist positiv (+0,41 auf der Log-Skala) und bestätigt den Aktions-Volumeneffekt, und der Preis-Spline trägt einen stark negativen linearen Trend (−1,71), das Kennzeichen einer fallenden Nachfragekurve.

In [3]:
PROZEDUR gam DATEN=store_sales PLOTS=components(CLM commonaxes);
   KLASSE Promotion;
   MODELL Units = PARAM(Promotion)
                 SPLINE(Price, df=4)
                 SPLINE(Temperature, df=4)
                 SPLINE(PromoSpend, df=3) / DIST=poisson METHOD=gcv;
   AUSGABE out=gam_fit predicted residual;
   BEZEICHNUNG Units="Verkaufte Einheiten" Price="Preis ($)"
         Temperature="Temperatur (°F)" PromoSpend="Werbeausgaben ($)"
         Promotion="Aktion";
AUSFÜHREN;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     Verkaufte Einheiten
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        43.592828
Null Deviance   2041.115451
AIC             254.611076

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                5.228805       0.000000       0.000000       0.000000
PROMOTIONYES               0.406972       0.000000       0.000000       0.000000
S(PRICE, DF = 4)          -1.705326       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.031495       0.000000       0.000000       0.000000
S(PROMOSPEND, DF = 3)       0.002307       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Preis ($))                4.0


NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Schritt 4 — Ein fokussiertes Preis- und Saisonalitätsmodell

Für eine schlankere Preisüberprüfung passen wir erneut an, nur mit den zwei entscheidungsrelevantesten glatten Treibern — Preis und Temperatur —, wobei wir dem Preis zusätzliche Flexibilität geben (`df=5`), um einen etwaigen Knick nahe einer psychologischen Preisschwelle aufzulösen. Das Aktions-Kennzeichen bleibt als parametrische Kontrolle erhalten.

Das Weglassen der Werbeausgaben erhöht die **Devianz auf 62,80** und den **AIC auf 269,82**, beide höher als die 43,59 und 254,61 des vollständigen Modells. Der parametrische `PROMOTIONYES`-Term absorbiert hier auch mehr vom Aktionssignal (+1,73 gegenüber +0,41), da der Ausgaben-Spline nicht mehr vorhanden ist, um es zu tragen. Der Preis-Spline behält seinen negativen Trend (−1,51), sodass die Kernnachfragegeschichte über die Spezifikationen hinweg stabil bleibt.

In [4]:
PROZEDUR gam DATEN=store_sales PLOTS=components(CLM);
   KLASSE Promotion;
   MODELL Units = PARAM(Promotion)
                 SPLINE(Price, df=5)
                 SPLINE(Temperature, df=4) / DIST=poisson;
   BEZEICHNUNG Units="Verkaufte Einheiten" Price="Preis ($)"
         Temperature="Temperatur (°F)" Promotion="Aktion";
AUSFÜHREN;



                                                   The GAM Procedure                                                    

Model Information
Response Variable     Verkaufte Einheiten
Distribution          poisson
Link Function         log
Number of Observations     100

Fit Statistics
Deviance        62.803733
Null Deviance   2041.115451
AIC             269.821548

Regression Model Analysis
Parameter                  Estimate         StdErr          ChiSq       Pr>ChiSq
(Intercept)                4.915205       0.000000       0.000000       0.000000
PROMOTIONYES               1.725573       0.000000       0.000000       0.000000
S(PRICE, DF = 5)          -1.511509       0.000000       0.000000       0.000000
S(TEMPERATURE, DF = 4)       0.027370       0.000000       0.000000       0.000000

Smoothing Model Analysis
Component                            DF            EDF
Spline(Preis ($))                5.0000         5.0000
Spline(Temperatur (°F))          4.0000         4.0000





NOTE: PROC GAM data=store_sales

NOTE: GAM wrapper backend: using R wrapper (gam::gam / mgcv::gam).
NOTE: PROC GAM completed.


## Interpretation der Ergebnisse

Die Tabelle **Regression Model Analysis** berichtet den linearen Trend innerhalb jeder Komponente plus den parametrischen Aktionseffekt. Der positive `PROMOTIONYES`-Koeffizient (+0,41 im vollständigen Modell, +1,73 im schlankeren) bestätigt die erwartete Aktions-Volumenwirkung, während der negative lineare Trend beim Preis-Spline (−1,71 und −1,51) die klassische fallende Nachfrage widerspiegelt. Der kleine positive lineare Term der Temperatur-Spline (+0,03) ist erwartet: Ihre eigentliche Geschichte ist die Krümmung um den Komfort-Höhepunkt bei 72 °F, die ein einzelner linearer Koeffizient nicht zusammenfassen kann.

Die Tabelle **Smoothing Model Analysis** berichtet die Freiheitsgrade, die jeder Spline verbraucht. Preis und Temperatur ziehen je 4 effektive Freiheitsgrade (5 für Preis im schlankeren Modell) und Werbeausgaben 3 — jeweils deutlich über dem einen Freiheitsgrad, den eine Gerade verwenden würde, weshalb eine lineare Regression diese gekrümmten Zusammenhänge übersehen würde.

Die **Fit Statistics** erlauben den direkten Vergleich der beiden Spezifikationen. Das vollständige Vier-Treiber-Modell weist eine Devianz von 43,59 und einen AIC von 254,61 gegenüber 62,80 und 269,82 des schlankeren Modells auf; beide Kriterien bevorzugen das vollständige Modell, was zeigt, dass Werbeausgaben und das Aktions-Kennzeichen Erklärungskraft über Preis und Temperatur hinaus hinzufügen. Relativ zur Null-Devianz von 2041,12 erfassen beide Modelle die überwältigende Mehrheit der Nachfragevariation.

Zusammen ergeben diese Tabellen einem Kategoriemanager eine quantifizierte, datengetriebene Nachfragegeschichte: eine steile Preisreaktion, die die Tiefe von Preisnachlässen informiert, ein saisonales Temperaturfenster und eine Werbeausgaben-Wirkung mit abnehmenden Grenzerträgen — deutlich schärfere Orientierung als eine einzelne lineare Elastizitätsschätzung. (PROC GAM akzeptiert auch `plots=components`, um die partiellen Vorhersagekurven für jeden glatten Term darzustellen; die numerischen Komponententabellen oben sind die Quelle, aus der diese Kurven gezeichnet werden.)